# 05 — GraphSAGE + LSTM Baseline (MCMIPF on-the-fly)

## Purpose
Train a baseline spatio-temporal model:
- Spatial encoder: GraphSAGE over a PATCH×PATCH graph
- Temporal encoder: LSTM over the history window
- Target: GHI at t_target (6h ahead)

## Import and settings

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import time
import os
import random
from copy import deepcopy

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("..").resolve()
DATASET_ROOT = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


## manifests

### Reproducibility

In [2]:
SITE = "uniandes"  # change to "elpaso" when needed

SITE_DIR = DATASET_ROOT / SITE
assert SITE_DIR.exists(), f"Missing dataset directory: {SITE_DIR}"

with open(SITE_DIR / "dataset_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

In [3]:
SEED = int(meta.get("seed", 42))

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Determinism settings (can reduce speed, but improves reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("SEED:", SEED)

SEED: 42


### Choose sites and load

In [4]:
print("Loaded dataset_meta.json:")
print(json.dumps(meta, indent=2))

MCMIPF_ROOT = Path(meta["mcmipf_root"])
FREQ_MIN = int(meta["freq_min"])
H = int(meta["horizon_steps"])
L = int(meta["history_steps"])

GRID_SIZE = int(meta["grid_size"])
SITE_CENTER = (int(meta["site_center_pix"]["row"]), int(meta["site_center_pix"]["col"]))

PATCH = 16#32
HALF = PATCH // 2

train_man = pd.read_parquet(SITE_DIR / "manifest_train.parquet")
val_man   = pd.read_parquet(SITE_DIR / "manifest_val.parquet")
test_man  = pd.read_parquet(SITE_DIR / "manifest_test.parquet")

print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

Loaded dataset_meta.json:
{
  "site": "uniandes",
  "grid_size": 256,
  "site_center_pix": {
    "row": 160,
    "col": 49
  },
  "freq_min": 10,
  "horizon_steps": 36,
  "history_steps": 12,
  "mcmipf_root": "/srv/projects/Proyecto_e_ladino/data_processed/GOES_v2/MCMIPF",
  "notes": "Manifest-only dataset. Satellite patches are extracted on-the-fly by the model."
}
train: (54508, 5)
val:   (12407, 5)
test:  (12075, 5)


### Debug

In [5]:
DEBUG = False  # True = quick sanity runs
DEBUG_TRAIN_N = 4000
DEBUG_VAL_N   = 1200
DEBUG_TEST_N  = 1200

if DEBUG:
    train_man = train_man.sample(n=min(DEBUG_TRAIN_N, len(train_man)), random_state=SEED).reset_index(drop=True)
    val_man   = val_man.sample(n=min(DEBUG_VAL_N, len(val_man)), random_state=SEED).reset_index(drop=True)
    test_man  = test_man.sample(n=min(DEBUG_TEST_N, len(test_man)), random_state=SEED).reset_index(drop=True)

print("DEBUG:", DEBUG)
print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

DEBUG: False
train: (54508, 5)
val:   (12407, 5)
test:  (12075, 5)


## Baseline: Persistence on the same manifests
Uses y_hat(t+H) = ghi(t)

In [6]:
GROUND_DIR = PROJECT_ROOT / "data" / "ground_aligned"
ground_path = GROUND_DIR / f"ground_10min_utc_{SITE}.parquet"
assert ground_path.exists(), f"Missing ground parquet for {SITE}: {ground_path}"

ground = pd.read_parquet(ground_path)
assert "ghi" in ground.columns, "Ground dataset missing 'ghi'"
assert str(ground.index.tz) == "UTC", "Ground index must be UTC"

def eval_persistence(manifest: pd.DataFrame, ground_df: pd.DataFrame) -> dict:
    t_label = pd.to_datetime(manifest["t_label"], utc=True)
    y_true = manifest["y"].astype(float).to_numpy()
    y_hat = ground_df.reindex(t_label)["ghi"].to_numpy()  # persistence: y(t)

    mask = np.isfinite(y_true) & np.isfinite(y_hat)
    y_true = y_true[mask]
    y_hat = y_hat[mask]

    rmse = float(np.sqrt(np.mean((y_true - y_hat) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_hat)))
    return {"n": int(mask.sum()), "rmse": rmse, "mae": mae}

print("=== Persistence baseline (same manifests) ===")
print("train:", eval_persistence(train_man, ground))
print("val:  ", eval_persistence(val_man, ground))
print("test: ", eval_persistence(test_man, ground))

=== Persistence baseline (same manifests) ===
train: {'n': 54470, 'rmse': 419.8303181060779, 'mae': 281.04479528180656}
val:   {'n': 12406, 'rmse': 378.32913327097435, 'mae': 247.05014589714656}
test:  {'n': 12075, 'rmse': 389.517844176586, 'mae': 250.79533714285714}


### Target normalization

In [7]:
y_train = train_man["y"].astype(float).to_numpy()
Y_MEAN = float(np.mean(y_train))
Y_STD  = float(np.std(y_train) + 1e-6)

def norm_y(y: float) -> float:
    return (y - Y_MEAN) / Y_STD

def denorm_y(arr: np.ndarray) -> np.ndarray:
    return arr * Y_STD + Y_MEAN

print("Target stats from train:")
print("  mean:", Y_MEAN)
print("  std: ", Y_STD)
print("  percentiles:", np.percentile(y_train, [0, 50, 90, 95, 99]).tolist())

Target stats from train:
  mean: 180.78839786820285
  std:  283.1064661153594
  percentiles: [0.0, 2.202, 628.0594000000003, 858.5459999999997, 1088.74274]


## Edge index

In [8]:
def build_edge_index_8n(patch: int) -> torch.Tensor:
    edges = []
    def node_id(rr: int, cc: int) -> int:
        return rr * patch + cc

    for rr in range(patch):
        for cc in range(patch):
            u = node_id(rr, cc)
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if dr == 0 and dc == 0:
                        continue
                    r2 = rr + dr
                    c2 = cc + dc
                    if 0 <= r2 < patch and 0 <= c2 < patch:
                        v = node_id(r2, c2)
                        edges.append((u, v))

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()  # (2, E)
    return edge_index

EDGE_INDEX = build_edge_index_8n(PATCH)
print("EDGE_INDEX:", EDGE_INDEX.shape)


EDGE_INDEX: torch.Size([2, 1860])


## MCMPIF 
Loand and extract Patch

In [9]:
# RX = re.compile(r"(?P<ymd>\d{8})_(?P<h>\d{2})_MCMIPF\.npz$")

def hour_path_for_timestamp(t: pd.Timestamp) -> Path:
    ymd = t.strftime("%Y%m%d")
    hh = t.strftime("%H")
    year = t.strftime("%Y")
    month = t.strftime("%m")
    fname = f"{ymd}_{hh}_MCMIPF.npz"
    path = MCMIPF_ROOT / year / month / fname
    return path

def slot_for_timestamp(t: pd.Timestamp) -> int:
    return int(t.strftime("%M")) // 10  # 0..5

def extract_patch(frame_c_hw: np.ndarray, center_rc: tuple[int, int], patch: int) -> np.ndarray:
    r0, c0 = center_rc
    half = patch // 2
    r1, r2 = r0 - half, r0 + half
    c1, c2 = c0 - half, c0 + half

    if (r1 < 0) or (c1 < 0) or (r2 > GRID_SIZE) or (c2 > GRID_SIZE):
        raise ValueError(f"Patch exceeds bounds: rows [{r1},{r2}) cols [{c1},{c2})")

    return frame_c_hw[:, r1:r2, c1:c2]  # (C, P, P)

def patch_to_node_features(patch_c_pp: np.ndarray) -> np.ndarray:
    # (C, P, P) -> (P*P, C)
    C, P1, P2 = patch_c_pp.shape
    x = np.transpose(patch_c_pp, (1, 2, 0)).reshape(P1 * P2, C).astype(np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x


## Helpers

In [10]:
from functools import lru_cache

@lru_cache(maxsize=128)  # tune: 256 hour-files in cache
def load_hour_npz(path_str: str) -> np.ndarray:
    path = Path(path_str)
    with np.load(path) as data:
        arr = data["mcmipf"].astype(np.float32)  # (6,16,256,256)
    return arr

In [11]:
class GraphSeqDataset(Dataset):
    def __init__(self, manifest: pd.DataFrame, site_center: tuple[int,int]):
        self.man = manifest.reset_index(drop=True)
        self.site_center = site_center

    def __len__(self) -> int:
        return len(self.man)

    def __getitem__(self, i: int):
        row = self.man.iloc[i]
        # y = float(row["y"])
        y = norm_y(float(row["y"]))
        history_ts = row["history_ts"]

        # history_ts is stored as list of strings
        if isinstance(history_ts, str):
            # parquet sometimes stores lists as strings depending on engine
            # expect JSON-like string
            history_ts = json.loads(history_ts)

        xs = []
        for ts_str in history_ts:
            # t = pd.Timestamp(ts_str).tz_localize("UTC") if "+" not in ts_str else pd.Timestamp(ts_str)
            # t = pd.Timestamp(ts_str)
            # if t.tzinfo is None:
            #     t = t.tz_localize("UTC")
            # else:
            #     t = t.tz_convert("UTC")

            # t = t.tz_convert("UTC")
            t = pd.to_datetime(ts_str, utc=True)

            p = hour_path_for_timestamp(t)
            slot = slot_for_timestamp(t)

            # data = np.load(p)
            # tensor = data["mcmipf"]  # (6, 16, 256, 256)
            # frame = tensor[slot]     # (16, 256, 256)
            
            # with np.load(p) as data:
            #     tensor = data["mcmipf"]
            #     frame = tensor[slot]

            tensor = load_hour_npz(str(p))
            frame = tensor[slot]

            patch = extract_patch(frame, self.site_center, PATCH)  # (16, 32, 32)
            x = patch_to_node_features(patch)                      # (1024, 16)

            xs.append(x)

        x_seq = np.stack(xs, axis=0)  # (L, N, C)
        return torch.from_numpy(x_seq), torch.tensor(y, dtype=torch.float32)

train_ds = GraphSeqDataset(train_man, SITE_CENTER)
val_ds   = GraphSeqDataset(val_man, SITE_CENTER)
test_ds  = GraphSeqDataset(test_man, SITE_CENTER)

print("train_ds:", len(train_ds), "val_ds:", len(val_ds), "test_ds:", len(test_ds))


train_ds: 54508 val_ds: 12407 test_ds: 12075


### Metrics

In [12]:
def metrics_from_arrays(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = y_true.astype(np.float64)
    y_pred = y_pred.astype(np.float64)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    return {"n": int(mask.sum()), "rmse": rmse, "mae": mae}

# @torch.no_grad()
# def eval_model(loader: DataLoader) -> dict:
#     model.eval()
#     ys, yhats = [], []
#     for x_seq, y in loader:
#         x_seq = x_seq.to(DEVICE)
#         y = y.to(DEVICE)
#         yhat = model(x_seq, EDGE_INDEX)
#         ys.append(y.detach().cpu().numpy())
#         yhats.append(yhat.detach().cpu().numpy())

#     y = np.concatenate(ys)
#     yhat = np.concatenate(yhats)
#     return metrics_from_arrays(y, yhat)

@torch.no_grad()
def eval_model(loader: DataLoader) -> dict:
    model.eval()
    ys, yhats = [], []
    for x_seq, y in loader:
        x_seq = x_seq.to(DEVICE)
        y = y.to(DEVICE)
        yhat = model(x_seq, EDGE_INDEX)
        ys.append(y.detach().cpu().numpy())
        yhats.append(yhat.detach().cpu().numpy())

    y = np.concatenate(ys)
    yhat = np.concatenate(yhats)

    # denormalize back to physical units
    y = denorm_y(y)
    yhat = denorm_y(yhat)

    return metrics_from_arrays(y, yhat)


## Data Load

In [13]:
BATCH_SIZE = 8#2
NUM_WORKERS = 4#0

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE=="cuda"))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE=="cuda"))
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE=="cuda"))


## GraphSage
custom-made -> no external libs

In [14]:
class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_nei  = nn.Linear(in_dim, out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # x: (N, F), edge_index: (2, E) with edges u->v
        N = x.size(0)
        src, dst = edge_index[0], edge_index[1]  # u, v

        nei_sum = torch.zeros((N, x.size(1)), device=x.device, dtype=x.dtype)
        nei_cnt = torch.zeros((N, 1), device=x.device, dtype=x.dtype)

        nei_sum.index_add_(0, dst, x[src])
        nei_cnt.index_add_(0, dst, torch.ones((dst.numel(), 1), device=x.device, dtype=x.dtype))

        nei_mean = nei_sum / (nei_cnt + 1e-6)

        out = self.lin_self(x) + self.lin_nei(nei_mean)
        return torch.relu(out)


## Model
**GraphSAGE per frame + LSTM over time**

In [15]:
class GraphSAGE_LSTM(nn.Module):
    def __init__(self, in_dim: int, hidden_g: int, hidden_t: int, out_dim: int = 1):
        super().__init__()
        self.g1 = GraphSAGELayer(in_dim, hidden_g)
        self.g2 = GraphSAGELayer(hidden_g, hidden_g)

        # temporal encoder over pooled graph embeddings
        self.lstm = nn.LSTM(input_size=hidden_g, hidden_size=hidden_t, batch_first=True)

        self.head = nn.Sequential(
            nn.Linear(hidden_t, hidden_t),
            nn.ReLU(),
            nn.Linear(hidden_t, out_dim)
        )

    def forward(self, x_seq: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # x_seq: (B, L, N, C)
        B, Ls, N, C = x_seq.shape
        edge_index = edge_index.to(x_seq.device)

        # process each time step
        embeds = []
        for t in range(Ls):
            x = x_seq[:, t]  # (B, N, C)

            # apply graph encoder per sample
            # (loop over batch to keep implementation simple and correct)
            pooled = []
            for b in range(B):
                xb = x[b]                  # (N, C)
                h = self.g1(xb, edge_index)
                h = self.g2(h, edge_index)
                g = h.mean(dim=0)          # global mean pooling -> (hidden_g,)
                pooled.append(g)
            pooled = torch.stack(pooled, dim=0)  # (B, hidden_g)
            embeds.append(pooled)

        z = torch.stack(embeds, dim=1)  # (B, L, hidden_g)
        out, _ = self.lstm(z)           # (B, L, hidden_t)
        last = out[:, -1]               # (B, hidden_t)
        yhat = self.head(last).squeeze(-1)  # (B,)
        return yhat


In [16]:
model = GraphSAGE_LSTM(in_dim=16, hidden_g=64, hidden_t=64).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
print("Model params:", sum(p.numel() for p in model.parameters())/1e6, "M")

Model params: 0.048001 M


### Sanity check

In [17]:
row = train_man.iloc[0]
history_ts = row["history_ts"]
if isinstance(history_ts, str):
    history_ts = json.loads(history_ts)

t0 = time.time()
for ts_str in history_ts[:10]:
    t = pd.to_datetime(ts_str, utc=True)
    p = hour_path_for_timestamp(t)
    tensor = load_hour_npz(str(p))
    _ = tensor[slot_for_timestamp(t)]
print("10 frames load time (s):", time.time() - t0)

10 frames load time (s): 0.06879281997680664


## Training

In [ ]:
EPOCHS = 30               
PATIENCE = 5              
MIN_DELTA = 2.0           
BEST_PATH = SITE_DIR / "best_model.pt"

train_log = []
best_rmse = float("inf")
best_epoch = 0
bad_epochs = 0

t_train0 = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_losses = []

    for x_seq, y in train_loader:
        x_seq = x_seq.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        yhat = model(x_seq, EDGE_INDEX)
        loss = loss_fn(yhat, y)
        loss.backward()
        opt.step()

        tr_losses.append(loss.item())

    # --- Validation (global metrics) ---
    val_metrics = eval_model(val_loader)
    val_rmse = float(val_metrics["rmse"])
    val_mae  = float(val_metrics["mae"])

    epoch_out = {
        "epoch": epoch,
        "train_mse": float(np.mean(tr_losses)),
        "val_rmse": val_rmse,
        "val_mae": val_mae,
        "epoch_seconds": float(time.time() - t0),
    }
    train_log.append(epoch_out)

    improved = (best_rmse - val_rmse) > MIN_DELTA

    if improved:
        best_rmse = val_rmse
        best_epoch = epoch
        bad_epochs = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": opt.state_dict(),
                "best_rmse": best_rmse,
                "meta": {
                    "site": SITE,
                    "patch": PATCH,
                    "L": L,
                    "H": H,
                    "seed": SEED,
                },
            },
            BEST_PATH,
        )
    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_mse={epoch_out['train_mse']:.5f} | "
        f"val_rmse={val_rmse:.5f} | val_mae={val_mae:.5f} | "
        f"time={epoch_out['epoch_seconds']:.1f}s | "
        f"best_rmse={best_rmse:.5f} (epoch {best_epoch}) | "
        f"bad={bad_epochs}/{PATIENCE}"
    )

    if bad_epochs >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}. Best epoch was {best_epoch} with RMSE={best_rmse:.5f}")
        break

total_train_seconds = float(time.time() - t_train0)
print("Training finished. Total seconds:", round(total_train_seconds, 1))
print("Best checkpoint:", BEST_PATH)

Epoch 01 | train_mse=1.00074 | val_rmse=258.88636 | val_mae=205.24753 | time=2654.3s | best_rmse=258.88636 (epoch 1) | bad=0/5


In [ ]:
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()

print("Loaded best model from epoch:", ckpt["epoch"], "| best_rmse:", ckpt["best_rmse"])

final_val = eval_model(val_loader)
final_test = eval_model(test_loader)

print("Best-ckpt Model val:", final_val)
print("Best-ckpt Model test:", final_test)


## Evaluation (val and test)

In [ ]:
# final_val = eval_model(val_loader)
# final_test = eval_model(test_loader)

# print("Model val:", final_val)
# print("Model test:", final_test)

Model val: {'n': 12407, 'rmse': 258.99662736218306, 'mae': 206.21237237776106}
Model test: {'n': 12075, 'rmse': 262.49626107155257, 'mae': 204.97573368933382}


## Summary 

In [ ]:
baseline_train = eval_persistence(train_man, ground)
baseline_val   = eval_persistence(val_man, ground)
baseline_test  = eval_persistence(test_man, ground)

summary = {
    "site": SITE,
    "device": DEVICE,
    "seed": SEED,
    "debug": {
        "enabled": bool(DEBUG),
        "train_n": int(len(train_man)),
        "val_n": int(len(val_man)),
        "test_n": int(len(test_man)),
    },
    "data_paths": {
        "site_dir": str(SITE_DIR),
        "mcmipf_root": str(MCMIPF_ROOT),
        "ground_path": str(ground_path),
    },
    "temporal": {
        "freq_min": int(FREQ_MIN),
        "history_steps_L": int(L),
        "horizon_steps_H": int(H),
        "history_hours": float(L * FREQ_MIN / 60.0),
        "horizon_hours": float(H * FREQ_MIN / 60.0),
    },
    "spatial": {
        "grid_size": int(GRID_SIZE),
        "patch": int(PATCH),
        "site_center_rc": {"row": int(SITE_CENTER[0]), "col": int(SITE_CENTER[1])},
        "nodes": int(PATCH * PATCH),
        "edge_index_shape": [int(EDGE_INDEX.shape[0]), int(EDGE_INDEX.shape[1])],
    },
    "target_norm": {
        "y_mean_train": float(Y_MEAN),
        "y_std_train": float(Y_STD),
        "y_percentiles_train": np.percentile(y_train, [0, 50, 90, 95, 99]).tolist(),
    },
    "baselines": {
        "persistence_train": baseline_train,
        "persistence_val": baseline_val,
        "persistence_test": baseline_test,
    },
    "model": {
        "arch": "GraphSAGE(2 layers) + mean pool + LSTM + MLP head",
        "in_dim": 16,
        "hidden_g": 64,
        "hidden_t": 64,
        "optimizer": "Adam",
        "lr": 1e-3,
        "batch_size": int(BATCH_SIZE),
        "num_workers": int(NUM_WORKERS),
        "epochs": int(EPOCHS),
        "train_seconds_total": float(total_train_seconds),
        "final_val": final_val,
        "final_test": final_test,
        "best_ckpt_path": str(BEST_PATH),
        "best_epoch": int(best_epoch),
        "best_rmse": float(best_rmse),

    },
    "training_log": train_log,
}

print("=== Baseline vs Model (val/test) ===")
print("Persistence val :", baseline_val)
print("Model val       :", final_val)
print("Persistence test:", baseline_test)
print("Model test      :", final_test)

print("\n=== Full run summary JSON ===")
print(json.dumps(summary, indent=2))

=== Baseline vs Model (val/test) ===
Persistence val : {'n': 12406, 'rmse': 378.32913327097435, 'mae': 247.05014589714656}
Model val       : {'n': 12407, 'rmse': 258.99662736218306, 'mae': 206.21237237776106}
Persistence test: {'n': 12075, 'rmse': 389.517844176586, 'mae': 250.79533714285714}
Model test      : {'n': 12075, 'rmse': 262.49626107155257, 'mae': 204.97573368933382}

=== Full run summary JSON ===
{
  "site": "uniandes",
  "device": "cuda",
  "seed": 42,
  "debug": {
    "enabled": false,
    "train_n": 54508,
    "val_n": 12407,
    "test_n": 12075
  },
  "data_paths": {
    "site_dir": "/srv/projects/Proyecto_e_ladino/data/datasets/manifest_v1/uniandes",
    "mcmipf_root": "/srv/projects/Proyecto_e_ladino/data_processed/GOES_v2/MCMIPF",
    "ground_path": "/srv/projects/Proyecto_e_ladino/data/ground_aligned/ground_10min_utc_uniandes.parquet"
  },
  "temporal": {
    "freq_min": 10,
    "history_steps_L": 24,
    "horizon_steps_H": 36,
    "history_hours": 4.0,
    "horizon_h

In [ ]:
RUNS_DIR = PROJECT_ROOT / "runs" / "graphsage_lstm"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

run_name = f"{SITE}_H{H}_L{L}_P{PATCH}_seed{SEED}_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}"
out_path = RUNS_DIR / f"{run_name}.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved run summary:", out_path)